In [1]:
import geopandas as gpd
from pathlib import Path

dataset_dir = Path.cwd().parent / "add_properties_to_geojson"
dataset_name = "vineyards_enriched_location.geojson"
dataset_path = dataset_dir / dataset_name
print(f".geojson path = {dataset_path}")

gdf = gpd.read_file(dataset_path)


.geojson path = d:\PycharmProjects\Vitis-AI\data\OSM_data\add_properties_to_geojson\vineyards_enriched_location.geojson


In [2]:
gdf.head(2)

,osm_id,osm_type,name,landuse,grape_variety,vine_row_orientation,vine_row_direction,crop,produce,plantation,...,wine:region,harvest_year_start,harvest_first_year,organic,irrigation,irrigation:water_supply,terraced,country,continent,geometry
0,28522677,way,NaN,vineyard,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Spain,Europe,"POLYGON ((-16.64421 28.16021, -16.64439 28.159..."
1,42125322,way,Clos d'Oranje,vineyard,syrah,NaN,NaN,grape,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,South Africa,Africa,"POLYGON ((18.41609 -33.94377, 18.41607 -33.943..."


# Оценка площадей

Спойлер, все вигня, там разброс боже мой

In [14]:
# gdf = gdf.to_crs(epsg=3857)  # CRS метрическая (например, UTM или EPSG:3857):
gdf_1 = gdf[['osm_id', 'geometry']]
gdf_1 = gdf_1.to_crs(epsg=3857)
gdf_1['area'] = gdf_1.geometry.area # кв.метры
gdf_1 = gdf_1[gdf_1['area'] > 0]
gdf_1.head(1)

,osm_id,geometry,area
0,28522677,"POLYGON ((-1852825.238 3269187.902, -1852845.3...",12931.287434


In [15]:
import numpy as np
from scipy import stats

areas = gdf_1['area'].values
mean_area = np.mean(areas)

alfa = 0.05
confidence = 1 - alfa
degrees_freedom = len(areas) - 1
standard_deviation = np.std(areas)
standard_error = stats.sem(areas) # sem это = СКО/sqrt(n)

# Интервал через t-распределение Стьюдента
ci = stats.t.interval(
    confidence,
    degrees_freedom,
    loc=mean_area,
    scale=standard_error
)

print(f"Средняя площадь: {mean_area:.2f}")
print(f"Стандартное отклонение: {standard_deviation:.2f}")
print(f"standard_error: {standard_error:.2f}")
print(f"95% Доверительный интервал: ({ci[0]:.2f}, {ci[1]:.2f})")

Средняя площадь: 62002.41
Стандартное отклонение: 418550.55
standard_error: 467.79
95% Доверительный интервал: (61085.55, 62919.27)


In [16]:
between_count = len([x for x in areas if ci[0] <= x <= ci[1]])
all_count = len(areas)
print(f"{between_count} / {all_count} = {between_count/all_count}")

3769 / 800549 = 0.004708019121877611
